## Search Engine with Tools and Agents

pip install wikipedia

pip install arxiv

#### inbuilt tools

In [7]:
from langchain_community.tools import WikipediaQueryRun,ArxivQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [8]:
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [9]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv.name

'arxiv'

In [10]:
tools=[wiki,arxiv]

#### Custom tools [RAG Based]

In [11]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [15]:
loader=WebBaseLoader("https://leetcode.com/")
doc=loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
final_doc=text_splitter.split_documents(doc)

embedding=HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
db=FAISS.from_documents(final_doc,embedding)

retriever=db.as_retriever()
retriever


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2914.07it/s]


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002675C80C410>, search_kwargs={})

In [18]:
from langchain_classic.tools.retriever import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"leetcode_tool","search any info related to dsa")

retriever_tool.name

'leetcode_tool'

In [20]:
tools=[wiki,arxiv,retriever_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\LangChain\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='leetcode_tool', description='search any info related to dsa', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000002675F5F5940>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000002675F5F5BC0>)]

#### Run the tools with Agents and LLM

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.3-70b-versatile")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002675F66FD70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002675F82FDA0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [37]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt=ChatPromptTemplate.from_messages([
    ("system","You are Helpful Assistant"),
    MessagesPlaceholder("agent_scratchpad"),
    ("user","{input}")
])



In [39]:
from langchain_classic.agents import create_openai_tools_agent
agent = create_openai_tools_agent(llm,tools,prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={'agent_scratchpad': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='

In [42]:
## to run agent

from langchain_classic.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent,tools=tools,verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={'agent_scratchpad': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typin

In [ ]:
res=agent_executor.invoke({"input":"tell me about leetcode"})



> Entering new AgentExecutor chain...

Invoking: `leetcode_tool` with `{'query': 'leetcode information'}`


Just a moment...Enable JavaScript and cookies to continue
Invoking: `leetcode_tool` with `{'query': 'leetcode info'}`


Just a moment...Enable JavaScript and cookies to continue
Invoking: `leetcode_tool` with `{'query': 'leetcode'}`


Just a moment...Enable JavaScript and cookies to continue
Invoking: `leetcode_tool` with `{'query': 'leetcode information'}`


Just a moment...Enable JavaScript and cookies to continue
Invoking: `leetcode_tool` with `{'query': 'what is leetcode'}`


Just a moment...Enable JavaScript and cookies to continueLeetCode is a popular online platform that provides a vast collection of coding challenges and interview practice problems for programmers and software developers. The platform offers a wide range of problems in various programming languages, including Python, Java, C++, JavaScript, and many others.

Here are some key features and benefits of Lee

{'input': 'tell me about leetcode',
 'output': 'LeetCode is a popular online platform that provides a vast collection of coding challenges and interview practice problems for programmers and software developers. The platform offers a wide range of problems in various programming languages, including Python, Java, C++, JavaScript, and many others.\n\nHere are some key features and benefits of LeetCode:\n\n1. **Large problem set**: LeetCode has an extensive library of over 1,500 problems, covering topics such as algorithms, data structures, database, and system design.\n2. **Practice and improvement**: The platform allows users to practice and improve their coding skills, problem-solving strategies, and interview preparation.\n3. **Interview practice**: LeetCode provides a realistic interview experience, with problems and questions similar to those asked in actual technical interviews.\n4. **Personalized progress tracking**: Users can track their progress, identify areas for improvement,

In [48]:
res=agent_executor.invoke({"input":"tell me all sliding window questionon leetcode with problem numbers "})



> Entering new AgentExecutor chain...

Invoking: `leetcode_tool` with `{'query': 'sliding window questions on LeetCode with problem numbers'}`


Just a moment...Enable JavaScript and cookies to continueHere are some common sliding window problems on LeetCode along with their problem numbers:

1. Maximum Subarray (53)
2. Minimum Window Substring (76)
3. Substring with Concatenation of All Words (30)
4. Permutation in String (567)
5. Sliding Window Maximum (239)
6. Longest Substring Without Repeating Characters (3)
7. Longest Substring with At Most K Distinct Characters (340)
8. Subarrays with K Different Integers (992)
9. Minimum Size Subarray Sum (209)
10. Maximum Average Subarray (643)
11. Longest Substring with Same Characters after Replacement (424)
12. Minimum Window Substring (76)
13. Substring with Concatenation of All Words (30)

These are some of the most common sliding window problems on LeetCode. However, it's worth noting that the sliding window technique can be applied to

In [51]:
res=agent.invoke({"input":"tell me all sliding window questionon leetcode with problem numbers "})

KeyError: 'intermediate_steps'